# Test pgvector + pg_textsearch

Quick connectivity check for the RAG database. It verifies:

1. A psql/psycopg connection to the `ragdb` database.
2. The **vector** extension works (create table, insert vectors, KNN search).
3. The **pg_textsearch** extension is loaded and its functions are available.

By default it connects to the in-cluster service DNS `pgvector`. If you run this
notebook from outside the cluster, set `PGHOST` to the LoadBalancer public IP.

## 1. Connect

In [ ]:
import os, psycopg

conn = psycopg.connect(
    host=os.getenv("PGHOST", "pgvector"),        # in-cluster DNS; use LoadBalancer IP if running outside the cluster
    port=int(os.getenv("PGPORT", "5432")),
    user=os.getenv("PGUSER", "raguser"),         # from 01-secret.yaml
    password=os.getenv("PGPASSWORD", "skk2026"), # from 01-secret.yaml
    dbname=os.getenv("PGDATABASE", "ragdb"),     # from 01-secret.yaml
)
conn.autocommit = True
cur = conn.cursor()

cur.execute("SELECT version();")
print(cur.fetchone()[0])

## 2. Check both extensions are installed

In [ ]:
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
cur.execute("CREATE EXTENSION IF NOT EXISTS pg_textsearch;")

cur.execute("""
    SELECT extname, extversion
    FROM pg_extension
    WHERE extname IN ('vector', 'pg_textsearch')
    ORDER BY extname;
""")
rows = cur.fetchall()
for name, ver in rows:
    print(f"{name:16} {ver}")

assert {r[0] for r in rows} == {"vector", "pg_textsearch"}, "an extension is missing!"
print("both extensions present")

## 3. Test `vector` — insert embeddings and run a nearest-neighbour search

In [ ]:
cur.execute("DROP TABLE IF EXISTS _conn_test_items;")
cur.execute("""
    CREATE TABLE _conn_test_items (
        id        bigserial PRIMARY KEY,
        content   text,
        embedding vector(3)
    );
""")
cur.execute("""
    INSERT INTO _conn_test_items (content, embedding) VALUES
        ('apple',  '[1, 1, 1]'),
        ('banana', '[1, 1, 0]'),
        ('car',    '[9, 0, 0]');
""")

query = "[1, 1, 1]"
cur.execute("""
    SELECT content, embedding <-> %s AS l2_distance
    FROM _conn_test_items
    ORDER BY embedding <-> %s
    LIMIT 3;
""", (query, query))

print("nearest to", query)
for content, dist in cur.fetchall():
    print(f"  {content:8} distance={dist:.4f}")

## 4. Test `pg_textsearch`

First confirm the library is preloaded (it must be in `shared_preload_libraries`,
set by the deployment args), then list the functions the extension provides so you
can see its available API.

In [ ]:
cur.execute("SHOW shared_preload_libraries;")
print("shared_preload_libraries:", cur.fetchone()[0])

cur.execute("""
    SELECT p.proname AS function,
           pg_get_function_identity_arguments(p.oid) AS args
    FROM pg_proc p
    JOIN pg_depend d    ON d.objid = p.oid AND d.deptype = 'e'
    JOIN pg_extension e ON e.oid = d.refobjid
    WHERE e.extname = 'pg_textsearch'
    ORDER BY 1;
""")
funcs = cur.fetchall()
print(f"
pg_textsearch exposes {len(funcs)} function(s):")
for fn, args in funcs:
    print(f"  {fn}({args})")

### Full-text search smoke test

A basic keyword search over the sample rows to confirm text search works end to end.
(This uses PostgreSQL's built-in `tsvector` matching; `pg_textsearch` adds BM25-style
ranking on top — use the functions listed above for that.)

In [ ]:
cur.execute("""
    SELECT content
    FROM _conn_test_items
    WHERE to_tsvector('english', content) @@ plainto_tsquery('english', %s);
""", ("banana",))
print("rows matching 'banana':", [r[0] for r in cur.fetchall()])

## 5. Clean up

In [ ]:
cur.execute("DROP TABLE IF EXISTS _conn_test_items;")
cur.close()
conn.close()
print("cleaned up, connection closed")